# DISCUS 6.23.01 File I/O

This notebook demonstrates reading and writing DISCUS `.cll` and `.stru` files.
See [docs/DISCUS_IO.md](../docs/DISCUS_IO.md) for the full documentation.

## The two kinds

| Kind | Command | Expands? | Example |
|---|---|---|---|
| `.cll` (unit-cell source) | `read cell file,n1,n2,n3` | Yes | `gaas.cll` (2 atoms) → 8/cell |
| `.stru` (saved structure) | `read stru file` | No | `gaas_222.stru` (64 atoms) → 64 |

In [1]:
from pathlib import Path
import os
import sys
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

from discuspdf import read_cll, read_stru, write_cll, write_stru, Lattice, Structure
import numpy as np

# Read the committed .cll fixture.
s = read_cll('examples/gaas.cll')
print(f'gaas.cll: {s.structure.n_atoms} atoms (unit cell)')
for i in range(s.structure.n_atoms):
    x, y, z = s.structure.frac[i]
    b = 0.0 if s.structure.b_iso is None else s.structure.b_iso[i]
    print(f'  {s.structure.species[i]:<4} {x:.6f} {y:.6f} {z:.6f} B={b:.6f}')

gaas.cll: 2 atoms (unit cell)
  GA   0.000000 0.000000 0.000000 B=1.000000
  AS   0.250000 0.250000 0.250000 B=1.000000


In [2]:
# Read a .stru fixture.
s2 = read_stru('tests/fixtures/discus_6_23_01/gaas_222.stru')
print(f'gaas_222.stru: {s2.structure.n_atoms} atoms')
from collections import Counter
print(f'species: {dict(Counter(s2.structure.species))}')

gaas_222.stru: 64 atoms
species: {'GA': 32, 'AS': 32}


In [3]:
# Build a 2×2×2 P1 GaAs Structure.
lat = Lattice.from_parameters(5.6537, 5.6537, 5.6537, 90.0, 90.0, 90.0)
unit_frac = np.array([
    [0.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.0, 0.5, 0.5], [0.5, 0.0, 0.5],
    [0.25, 0.25, 0.25], [0.75, 0.75, 0.25], [0.25, 0.75, 0.75], [0.75, 0.25, 0.75],
])
unit_species = np.array(['GA', 'GA', 'GA', 'GA', 'AS', 'AS', 'AS', 'AS'], dtype=object)
n = (2, 2, 2)
m = 4  # N / prod(n) = 32 / 8 = 4
N = 32
species = np.empty(N, dtype=object)
frac = np.empty((N, 3))
cell_index = np.zeros((N, 3), dtype=np.int64)
for r in range(N):
    cell_num = r // m
    slot = r % m
    iz = cell_num // 4
    rem = cell_num % 4
    iy = rem // 2
    ix = rem % 2
    cell_index[r] = (ix, iy, iz)
    species[r] = unit_species[slot]
    frac[r] = (unit_frac[slot] + np.array([ix, iy, iz])) / np.array(n)
s3 = Structure(lat, species, frac, model='configuration', b_iso=np.full(N, 0.35),
               cell_index=cell_index, supercell=n, pbc=(True, True, True), min_separation=None)
print(f'2×2×2 P1 GaAs: {s3.n_atoms} atoms')
print(f'species: {dict(Counter(s3.species))}')

2×2×2 P1 GaAs: 32 atoms
species: {'GA': 32}


In [4]:
# Write .cll (unit cell) and .stru (supercell) to a temporary directory.
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    unit_cell = Structure(lat, unit_species, unit_frac, model='configuration',
                          b_iso=np.full(8, 0.35), supercell=(1, 1, 1),
                          pbc=(True, True, True), min_separation=None)
    cll_path = Path(tmpdir) / 'gaas.cll'
    stru_path = Path(tmpdir) / 'gaas.stru'
    write_cll(cll_path, unit_cell)
    write_stru(stru_path, s3)

    # Read back.
    s_cll = read_cll(cll_path)
    s_stru = read_stru(stru_path)
    print(f'.cll read-back: {s_cll.structure.n_atoms} atoms (unit cell)')
    print(f'.stru read-back: {s_stru.structure.n_atoms} atoms (2×2×2)')

    # Print first 6 lines of each.
    print('\n--- gaas.cll ---')
    for line in cll_path.read_text().splitlines()[:6]:
        print(line)
    print('\n--- gaas.stru ---')
    for line in stru_path.read_text().splitlines()[:6]:
        print(line)

.cll read-back: 8 atoms (unit cell)
.stru read-back: 32 atoms (2×2×2)

--- gaas.cll ---
title structure
spcgr P1
cell 5.653700 5.653700 5.653700 90.000000 90.000000 90.000000
atoms
GA   0.000000 0.000000 0.000000 0.350000
GA   0.500000 0.500000 0.000000 0.350000

--- gaas.stru ---
title discuspdf structure
spcgr P1
cell    2.826850,  2.826850,  2.826850, 90.000000, 90.000000, 90.000000
ncell       2,       2,       2,       4
format numbers,XYZBPMMOS
atoms      x,              y,              z,             Biso,    Property,  MoleNo,  MoleAt,   Occ,     St,  Sh,  Sk,  Sl


## Live test (optional)

The following cell runs only if `DISCUS_LIVE` is set in the environment.

In [5]:
import os
if os.environ.get('DISCUS_LIVE') == '1':
    from tests.discus_live import run_discus
    import shutil
    workdir = os.environ.get('DISCUS_LIVE_WORKDIR', r'C:\tmp\a3live')
    os.makedirs(workdir, exist_ok=True)
    shutil.copy('examples/gaas.cll', os.path.join(workdir, 'gaas.cll'))
    macro = 'discus\nread\n  cell gaas.cll,2,2,2\neval n[1]\nexit\nexit\n'
    result = run_discus(macro, workdir)
    print('DISCUS live test:')
    print(f'  banner: {result.banner}')
    print(f'  atoms: 64 expected')
    assert '64.0000000000' in result.stdout
    print('  OK')
else:
    print('DISCUS_LIVE not set; skipping live test.')

DISCUS_LIVE not set; skipping live test.
